<a href="https://colab.research.google.com/github/t2oguns/FootballDefendersScript/blob/main/PremDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re

In [ ]:
df=pd.read_html("https://fbref.com/en/comps/9/schedule/Premier-League-Scores-and-Fixtures", attrs={"id": "sched_2024-2025_9_1"})[0]

In [ ]:
df

,Wk,Day,Date,Time,Home,xG,Score,xG.1,Away,Attendance,Venue,Referee,Match Report,Notes
0,1.0,Fri,2024-08-16,20:00,Manchester Utd,2.4,1–0,0.4,Fulham,73297.0,Old Trafford,Robert Jones,Match Report,NaN
1,1.0,Sat,2024-08-17,12:30,Ipswich Town,0.5,0–2,2.6,Liverpool,30014.0,Portman Road Stadium,Tim Robinson,Match Report,NaN
2,1.0,Sat,2024-08-17,15:00,Newcastle Utd,0.3,1–0,1.8,Southampton,52196.0,St James' Park,Craig Pawson,Match Report,NaN
3,1.0,Sat,2024-08-17,15:00,Everton,0.5,0–3,1.4,Brighton,39217.0,Goodison Park,Simon Hooper,Match Report,NaN
4,1.0,Sat,2024-08-17,15:00,Nott'ham Forest,1.3,1–1,1.2,Bournemouth,29763.0,The City Ground,Michael Oliver,Match Report,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
415,38.0,Sun,2025-05-25,16:00,Nott'ham Forest,1.2,0–1,1.1,Chelsea,30263.0,The City Ground,Anthony Taylor,Match Report,NaN
416,38.0,Sun,2025-05-25,16:00,Fulham,1.3,0–2,3.0,Manchester City,27671.0,Craven Cottage,Andy Madley,Match Report,NaN
417,38.0,Sun,2025-05-25,16:00,Newcastle Utd,1.2,0–1,1.2,Everton,52221.0,St James' Park,Tony Harrington,Match Report,NaN
418,38.0,Sun,2025-05-25,16:00,Bournemouth,1.6,2–0,0.3,Leicester City,11238.0,Vitality Stadium,Lewis Smith,Match Report,NaN


In [ ]:
player_name = "Adams"
player_team = "Bournemouth"

In [ ]:
df.rename(columns={'xG.1': 'xGA'}, inplace=True)

In [ ]:
combined_df = df

In [ ]:
combined_df['Date'] = pd.to_datetime(combined_df['Date'])

In [ ]:
combined_df.sort_values(by='Date', inplace=True)

In [ ]:
cleaned_df = combined_df.dropna(subset='Score')

cleaned_df = combined_df.dropna(subset='Day')

In [ ]:
player_df=pd.read_html("https://fbref.com/en/players/2b09d998/matchlogs/2024-2025/c9/Tyler-Adams-Match-Logs", attrs={"id": "matchlogs_2024-2025_9"})[0]

In [ ]:
team_link = "https://fbref.com/en/squads/4ba7cbea/2024-2025/matchlogs/c9/schedule/Bournemouth-Scores-and-Fixtures-Premier-League"

In [ ]:
team_df=pd.read_html(team_link, attrs={"id": "matchlogs_for"})[0]

In [ ]:
player_df.columns = [(col)[1] for col in player_df.columns.values]

In [ ]:
player_df.dropna(subset='Venue', inplace=True)

In [ ]:
team_df.dropna(subset='Venue', inplace=True)

In [ ]:
difference = team_df[~team_df['Date'].isin(player_df['Date'])]

In [ ]:
difference["Team"] = player_team

<ipython-input-458-e9d400f817e5>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  difference["Team"] = player_team


In [ ]:
%pip install google-colab-selenium

In [ ]:
import google_colab_selenium as gs
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup, Comment
import pandas as pd
import time

In [ ]:
# Setup headless Chrome
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = gs.Chrome(options=options)

<IPython.core.display.Javascript object>

In [ ]:
def get_game_links(table):
    links = []

    if table:
      for row in table.find_all("tr")[1:]:  # Skip header
          first_col = row.find("th")
          if first_col:
              a = first_col.find("a")

              if a and a.has_attr("href"):
                  links.append((a.text,a["href"]))

    return links

driver.get(team_link)

html = driver.page_source

soup = BeautifulSoup(html, "html.parser")

table = soup.find("table", {"id": "matchlogs_for"})


links = get_game_links(table)


In [ ]:
def mark_if_in(value, reference_set):
    return player_team+"_no_"+player_name.lower() if (value['Home'], value['Date'].strftime('%Y-%m-%d')) in reference_set else value['Home']

# Create a set of reference values
reference_teams = set(zip(difference['Team'], difference['Date']))

# Apply marker to the 'Team' column
cleaned_df['Home'] = cleaned_df.transpose().apply(lambda x: mark_if_in(x, reference_teams))

<ipython-input-463-f6bcadba1083>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df['Home'] = cleaned_df.transpose().apply(lambda x: mark_if_in(x, reference_teams))


In [ ]:
def mark_if_in_away(value, reference_set):
    return player_team+"_no_"+player_name.lower() if (value['Away'], value['Date'].strftime('%Y-%m-%d')) in reference_set else value['Away']

# Create a set of reference values
reference_teams = set(zip(difference['Team'], difference['Date']))

# Apply marker to the 'Team' column
cleaned_df['Away'] = cleaned_df.transpose().apply(lambda x: mark_if_in_away(x, reference_teams))

<ipython-input-464-e963079eed18>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df['Away'] = cleaned_df.transpose().apply(lambda x: mark_if_in_away(x, reference_teams))


In [ ]:
cleaned_df = cleaned_df.reset_index(drop=True)

In [ ]:
player_df['Min'] = pd.to_numeric(player_df['Min'], errors='coerce')
player_df = player_df.dropna(subset=['Min'])

In [ ]:
def parse_minute(x):
    try:
        x = str(x)  # ensure string
        if '+' in x:
            parts = x.split('+')
            return int(parts[0]) + int(parts[1])
        else:
            return int(x)
    except:
        return None  # or np.nan if you import numpy as np

In [ ]:
cleaned_df['Minutes_player'] = 90

In [ ]:
new_df = cleaned_df.iloc[0:0].copy()
for index, row in player_df.dropna(subset='Min').iterrows():
    if(int(row['Min'])<90 and int(row['Min'])!=0):
      split_df=pd.read_html("https://fbref.com"+next(sub for sub in links if row['Date'] in sub)[1], attrs={"id": "shots_all"})[0]
      split_df.columns = [col[1] for col in split_df.columns]
      split_df.dropna(subset='Minute', inplace=True)
      split_df['Minute_x'] = split_df['Minute'].apply(parse_minute)

      goalsPlayer = [0,0]
      goalsNoPlayer = [0,0]

      not_player = pd.DataFrame()
      player = pd.DataFrame()

      team_name = player_df["Squad"].iloc[0]

      if(row['Start']=='N'):
        start_time = 90-int(row['Min'])
        not_player = split_df[pd.to_numeric(split_df['Minute_x']) < start_time]
        player = split_df[pd.to_numeric(split_df['Minute_x']) >= start_time]

      if(row['Start'].startswith('Y')):
        end_time = int(row['Min'])
        not_player = split_df[pd.to_numeric(split_df['Minute_x']) > end_time]
        player = split_df[pd.to_numeric(split_df['Minute_x']) <= end_time]

      goalsPlayer = [player[(player['Outcome'] == 'Goal') & (player['Squad'] == player_df["Squad"].iloc[0])].shape[0],
      player[(player['Outcome'] == 'Goal') & (player['Squad'] != player_df["Squad"].iloc[0])].shape[0]]

      goalsNoPlayer = [not_player[(not_player['Outcome'] == 'Goal') & (not_player['Squad'] == player_df["Squad"].iloc[0])].shape[0],
      not_player[(not_player['Outcome'] == 'Goal') & (not_player['Squad'] != player_df["Squad"].iloc[0])].shape[0]]

      dfIndex = cleaned_df[((cleaned_df['Date'] == row['Date']) & ((cleaned_df['Home'] == team_name) | (cleaned_df['Away'] == team_name)))].index[0]


      xGA = player[player['Squad'] != player_df["Squad"].iloc[0]]['xG'].sum()
      xG = player[player['Squad'] == player_df["Squad"].iloc[0]]['xG'].sum()

      notxG = not_player[not_player['Squad'] == player_df["Squad"].iloc[0]]['xG'].sum()
      notxGA = not_player[not_player['Squad'] != player_df["Squad"].iloc[0]]['xG'].sum()


      if (cleaned_df.loc[dfIndex]["Home"]!=player_df["Squad"].iloc[0]):
        xGA, xG = xG, xGA
        notxGA,notxG = notxG,notxGA

        goalsPlayer[0], goalsPlayer[1] = goalsPlayer[1], goalsPlayer[0]
        goalsNoPlayer[0], goalsNoPlayer[1] = goalsNoPlayer[1], goalsNoPlayer[0]


      cleaned_df.loc[dfIndex, 'xG'] = xG
      cleaned_df.loc[dfIndex, 'xGA'] = xGA
      cleaned_df.loc[dfIndex, 'Score'] = f"{goalsPlayer[0]}-{goalsPlayer[1]}"
      cleaned_df.loc[dfIndex, 'Minutes_player'] = int(row['Min'])

      new_row = cleaned_df.loc[dfIndex].copy()

      new_row['xG'] = notxG
      new_row['xGA'] = notxGA
      new_row['Score'] = f"{goalsNoPlayer[0]}-{goalsNoPlayer[1]}"
      new_row['Minutes_player'] = 90-int(row['Min'])

      if((new_row['Home']==player_df["Squad"].iloc[0]) or new_row['Away']== player_df["Squad"].iloc[0]):
        if(new_row['Home']==player_df["Squad"].iloc[0]):
          new_row['Home'] = player_team+"_no_"+player_name.lower()

        elif(new_row['Away']== player_df["Squad"].iloc[0]):
          new_row['Away'] = player_team+"_no_"+player_name.lower()

        new_df = pd.concat([new_df, pd.DataFrame(new_row).T], ignore_index=True)
      time.sleep(3)

/en/matches/bce300b2/Aston-Villa-Bournemouth-October-26-2024-Premier-League
/en/matches/ed970fef/Bournemouth-Manchester-City-November-2-2024-Premier-League
/en/matches/1c998ef5/Brentford-Bournemouth-November-9-2024-Premier-League
/en/matches/4708a5bf/Bournemouth-Brighton-and-Hove-Albion-November-23-2024-Premier-League
/en/matches/49cd674b/Bournemouth-West-Ham-United-December-16-2024-Premier-League
/en/matches/8381ba53/Manchester-United-Bournemouth-December-22-2024-Premier-League
/en/matches/886ca6b3/Bournemouth-Crystal-Palace-December-26-2024-Premier-League
/en/matches/43f0c302/Bournemouth-Everton-January-4-2025-Premier-League
/en/matches/ccdda6d3/Bournemouth-Wolverhampton-Wanderers-February-22-2025-Premier-League
/en/matches/f3480c01/Bournemouth-Brentford-March-15-2025-Premier-League
/en/matches/19c54be2/Bournemouth-Ipswich-Town-April-2-2025-Premier-League
/en/matches/9d095ebf/Crystal-Palace-Bournemouth-April-19-2025-Premier-League
/en/matches/dd48659a/Bournemouth-Aston-Villa-May-10-2

In [ ]:
cleaned_df

,Wk,Day,Date,Time,Home,xG,Score,xGA,Away,Attendance,Venue,Referee,Match Report,Notes,Minutes_player
0,1.0,Fri,2024-08-16,20:00,Manchester Utd,2.4,1–0,0.4,Fulham,73297.0,Old Trafford,Robert Jones,Match Report,NaN,90
1,1.0,Sat,2024-08-17,12:30,Ipswich Town,0.5,0–2,2.6,Liverpool,30014.0,Portman Road Stadium,Tim Robinson,Match Report,NaN,90
2,1.0,Sat,2024-08-17,15:00,Newcastle Utd,0.3,1–0,1.8,Southampton,52196.0,St James' Park,Craig Pawson,Match Report,NaN,90
3,1.0,Sat,2024-08-17,15:00,Everton,0.5,0–3,1.4,Brighton,39217.0,Goodison Park,Simon Hooper,Match Report,NaN,90
4,1.0,Sat,2024-08-17,15:00,Nott'ham Forest,1.3,1–1,1.2,Bournemouth_no_adams,29763.0,The City Ground,Michael Oliver,Match Report,NaN,90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,38.0,Sun,2025-05-25,16:00,Wolves,1.0,1–1,1.4,Brentford,31382.0,Molineux Stadium,Craig Pawson,Match Report,NaN,90
376,38.0,Sun,2025-05-25,16:00,Ipswich Town,0.7,1–3,1.1,West Ham,29771.0,Portman Road Stadium,Tim Robinson,Match Report,NaN,90
377,38.0,Sun,2025-05-25,16:00,Manchester Utd,2.9,2–0,0.4,Aston Villa,73839.0,Old Trafford,Thomas Bramall,Match Report,NaN,90
378,38.0,Sun,2025-05-25,16:00,Southampton,0.6,1–2,2.3,Arsenal,31289.0,St Mary's Stadium,Darren Bond,Match Report,NaN,90


In [ ]:
cleaned_df =pd.concat([cleaned_df, new_df], ignore_index=True)

In [ ]:
final_df = cleaned_df[['Date', 'Away','Home', 'Venue', 'xG', 'xGA', 'Minutes_player']].copy()

In [ ]:
final_df

,Date,Away,Home,Venue,xG,xGA,Minutes_player
0,2024-08-16 00:00:00,Fulham,Manchester Utd,Old Trafford,2.4,0.4,90
1,2024-08-17 00:00:00,Liverpool,Ipswich Town,Portman Road Stadium,0.5,2.6,90
2,2024-08-17 00:00:00,Southampton,Newcastle Utd,St James' Park,0.3,1.8,90
3,2024-08-17 00:00:00,Brighton,Everton,Goodison Park,0.5,1.4,90
4,2024-08-17 00:00:00,Bournemouth_no_adams,Nott'ham Forest,The City Ground,1.3,1.2,90
...,...,...,...,...,...,...,...
388,2025-02-22 00:00:00,Wolves,Bournemouth_no_adams,Vitality Stadium,0.83,1.28,59
389,2025-03-15 00:00:00,Brentford,Bournemouth_no_adams,Vitality Stadium,0.54,0.06,12
390,2025-04-02 00:00:00,Ipswich Town,Bournemouth_no_adams,Vitality Stadium,0.42,0.81,45
391,2025-04-19 00:00:00,Bournemouth_no_adams,Crystal Palace,Selhurst Park,0.24,0.23,30


In [ ]:
import numpy as np

In [ ]:
final_df.insert(loc=4, column='1', value=np.ones(len(final_df), dtype=int))

In [ ]:
final_df

,Date,Away,Home,Venue,1,xG,xGA,Minutes_player
0,2024-08-16 00:00:00,Fulham,Manchester Utd,Old Trafford,1,2.4,0.4,90
1,2024-08-17 00:00:00,Liverpool,Ipswich Town,Portman Road Stadium,1,0.5,2.6,90
2,2024-08-17 00:00:00,Southampton,Newcastle Utd,St James' Park,1,0.3,1.8,90
3,2024-08-17 00:00:00,Brighton,Everton,Goodison Park,1,0.5,1.4,90
4,2024-08-17 00:00:00,Bournemouth_no_adams,Nott'ham Forest,The City Ground,1,1.3,1.2,90
...,...,...,...,...,...,...,...,...
388,2025-02-22 00:00:00,Wolves,Bournemouth_no_adams,Vitality Stadium,1,0.83,1.28,59
389,2025-03-15 00:00:00,Brentford,Bournemouth_no_adams,Vitality Stadium,1,0.54,0.06,12
390,2025-04-02 00:00:00,Ipswich Town,Bournemouth_no_adams,Vitality Stadium,1,0.42,0.81,45
391,2025-04-19 00:00:00,Bournemouth_no_adams,Crystal Palace,Selhurst Park,1,0.24,0.23,30


In [ ]:
final_df.insert(loc=6, column='2', value=np.ones(len(final_df), dtype=int))

In [ ]:
def get_lastname(name):
    if type(name)==float:
        return name
    return list(name)[0]

In [ ]:
final_df.insert(loc=1, column='away_score', value=cleaned_df['Score'].str[-1])

In [ ]:
final_df.insert(loc=3, column='home_score', value=cleaned_df['Score'].str[0])

In [ ]:
final_df.reset_index(inplace=True)

In [ ]:
final_df = final_df.rename(columns={'Away': 'away_team', 'Home': 'home_team', 'Venue': 'location', 'xGA': 'away_xG', 'xG': 'home_xG'})

In [ ]:
final_df['1'] = final_df['1']*final_df['Minutes_player']/90
final_df['2'] = final_df['2']*final_df['Minutes_player']/90

In [ ]:
final_df

,index,Date,away_score,away_team,home_score,home_team,location,1,home_xG,2,away_xG,Minutes_player
0,0,2024-08-16 00:00:00,0,Fulham,1,Manchester Utd,Old Trafford,1.0,2.4,1.0,0.4,90
1,1,2024-08-17 00:00:00,2,Liverpool,0,Ipswich Town,Portman Road Stadium,1.0,0.5,1.0,2.6,90
2,2,2024-08-17 00:00:00,0,Southampton,1,Newcastle Utd,St James' Park,1.0,0.3,1.0,1.8,90
3,3,2024-08-17 00:00:00,3,Brighton,0,Everton,Goodison Park,1.0,0.5,1.0,1.4,90
4,4,2024-08-17 00:00:00,1,Bournemouth_no_adams,1,Nott'ham Forest,The City Ground,1.0,1.3,1.0,1.2,90
...,...,...,...,...,...,...,...,...,...,...,...,...
388,388,2025-02-22 00:00:00,1,Wolves,0,Bournemouth_no_adams,Vitality Stadium,0.655556,0.83,0.655556,1.28,59
389,389,2025-03-15 00:00:00,0,Brentford,0,Bournemouth_no_adams,Vitality Stadium,0.133333,0.54,0.133333,0.06,12
390,390,2025-04-02 00:00:00,1,Ipswich Town,0,Bournemouth_no_adams,Vitality Stadium,0.5,0.42,0.5,0.81,45
391,391,2025-04-19 00:00:00,0,Bournemouth_no_adams,0,Crystal Palace,Selhurst Park,0.333333,0.24,0.333333,0.23,30


In [ ]:
final_df=final_df.reindex(columns=['away_team', 'away_score', 'home_team', 'home_score', 'location', 'away_xG', '1', 'home_xG', '2'])

In [ ]:
final_df.dropna(subset='away_score', inplace=True)

In [ ]:
final_df

,away_team,away_score,home_team,home_score,location,away_xG,1,home_xG,2
0,Fulham,0,Manchester Utd,1,Old Trafford,0.4,1.0,2.4,1.0
1,Liverpool,2,Ipswich Town,0,Portman Road Stadium,2.6,1.0,0.5,1.0
2,Southampton,0,Newcastle Utd,1,St James' Park,1.8,1.0,0.3,1.0
3,Brighton,3,Everton,0,Goodison Park,1.4,1.0,0.5,1.0
4,Bournemouth_no_adams,1,Nott'ham Forest,1,The City Ground,1.2,1.0,1.3,1.0
...,...,...,...,...,...,...,...,...,...
388,Wolves,1,Bournemouth_no_adams,0,Vitality Stadium,1.28,0.655556,0.83,0.655556
389,Brentford,0,Bournemouth_no_adams,0,Vitality Stadium,0.06,0.133333,0.54,0.133333
390,Ipswich Town,1,Bournemouth_no_adams,0,Vitality Stadium,0.81,0.5,0.42,0.5
391,Bournemouth_no_adams,0,Crystal Palace,0,Selhurst Park,0.23,0.333333,0.24,0.333333


In [ ]:
from google.colab import drive

In [ ]:
drive.mount('drive')

Drive already mounted at drive; to attempt to forcibly remount, call drive.mount("drive", force_remount=True).


In [ ]:
final_df.to_csv(f'/content/drive/My Drive/{player_name.lower()}.csv')